In [ ]:
import torch
from diffusers import StableDiffusionXLPipeline
import os

model_id = "stabilityai/stable-diffusion-xl-base-1.0"
pipe = StableDiffusionXLPipeline.from_pretrained(
    model_id, 
    torch_dtype=torch.float16, 
    variant="fp16", 
    use_safetensors=True
)

# trun off the CPU 
pipe.enable_model_cpu_offload()

stones = ["diamond", "emerald", "sapphire"]
shapes = ["round cut", "princess cut", "cushion cut"] 
settings = ["yellow gold", "white gold", "rose gold"] # white gold > platinum


output_dir = r"./data/Jewelry_Dataset_SDXL"
os.makedirs(output_dir, exist_ok=True)

images_per_combo = 35


print("Starting local SDXL generation")
for stone in stones:
    for shape in shapes:
        for setting in settings:
            
            positive_prompt = f"Professional jewelry macro photography of a single ring. The ring features exactly one {shape} {stone} as the centerpiece. The band is made of {setting}. Pure white studio background, bright studio lighting. High resolution, extreme detail, photorealistic."
            negative_prompt = "hands, fingers, human anatomy, skin, multiple stones, scattered stones, plain band, no stone, text, watermark, dark background, blurry, deformed geometry, ugly"
            
            combo_name = f"{shape}_{stone}_{setting}".replace(" ", "_")
            combo_dir = os.path.join(output_dir, combo_name)
            os.makedirs(combo_dir, exist_ok=True)

            print(f"Checking images for: {combo_name}")

            for i in range(images_per_combo):
                file_name = f"{combo_name}_{i+1}.png"
                file_path = os.path.join(combo_dir, file_name)
                
                # The Resume Check: Look if the image already exists
                if os.path.exists(file_path):
                    print(f"  -> Skipping {file_name}, already exists.")
                    continue
                
                # Generate new image
                image = pipe(
                    prompt=positive_prompt, 
                    negative_prompt=negative_prompt, 
                    num_inference_steps=30
                ).images[0] 
                
                image.save(file_path)
                print(f"  -> Saved {file_name}")

print("Clean dataset generation complete!")

In [ ]:
import torch
from diffusers import StableDiffusionXLPipeline
import os

model_id = "stabilityai/stable-diffusion-xl-base-1.0"
pipe = StableDiffusionXLPipeline.from_pretrained(
    model_id, torch_dtype=torch.float16, variant="fp16", use_safetensors=True
)
pipe.enable_model_cpu_offload()

stones = ["diamond", "emerald", "sapphire"]
shapes = ["round cut", "princess cut", "cushion cut"] 
settings = ["yellow gold", "white gold", "rose gold"]

output_dir = r"C:\Users\hamza\OneDrive\Downloads\Selected topics\project\Jewelry_Dataset_SDXL"

images_per_combo = 35

print("Starting targeted Solitaire generation")
for stone in stones:
    for shape in shapes:
        for setting in settings:
            
            # The extreme "Solitaire-Forcing" prompt
            positive_prompt = f"Professional jewelry macro photography. A single ring featuring exactly one {shape} {stone} in a strict solitaire setting. The band is a plain, smooth, unadorned {setting} with absolutely ZERO side stones or diamonds on the band. Pure white studio background, photorealistic."
            
            # Aggressive negative prompt
            negative_prompt = "pave, side stones, halo, multiple stones, diamonds on band, hands, fingers, human anatomy, text, watermark"
            
            # Note the new "_solitaire" folder naming convention
            combo_name = f"{shape}_{stone}_{setting}_solitaire".replace(" ", "_")
            combo_dir = os.path.join(output_dir, combo_name)
            os.makedirs(combo_dir, exist_ok=True)

            print(f"Checking images for: {combo_name}")

            for i in range(images_per_combo):
                file_name = f"{combo_name}_{i+1}.png"
                file_path = os.path.join(combo_dir, file_name)
                
                if os.path.exists(file_path):
                    continue
                
                image = pipe(
                    prompt=positive_prompt, 
                    negative_prompt=negative_prompt, 
                    num_inference_steps=30
                ).images[0] 
                
                image.save(file_path)
                print(f"  -> Saved {file_name}")

print("Solitaire dataset complete!")

In [ ]:
import os
import json

dataset_dir = r"C:\Users\hamza\OneDrive\Downloads\Selected topics\project\Jewelry_Dataset_SDXL"
metadata_file = os.path.join(dataset_dir, "metadata.jsonl")


with open(metadata_file, 'w') as f:
    # Loop through all 54 folders
    for folder_name in os.listdir(dataset_dir):
        folder_path = os.path.join(dataset_dir, folder_name)
        
        # Skip if its not a directory
        if not os.path.isdir(folder_path):
            continue

        components = folder_name.replace("_", " ").split()

        clean_prompt = f"A professional jewelry macro photograph of a {folder_name.replace('_', ' ')} ring, pure white background, highly detailed"


        for image_name in os.listdir(folder_path):
            if image_name.endswith(".png"):
                # The exact JSON structure Hugging Face Diffusers expects
                metadata_entry = {
                    "file_name": f"{folder_name}/{image_name}",
                    "text": clean_prompt
                }
                f.write(json.dumps(metadata_entry) + '\n')

print(f" Metadata saved to {metadata_file}")